# WarpKriging with categorical warping (Octave)

The **categorical** warp embeds each level $\ell \in \{0, \ldots, L-1\}$ into $\mathbb{R}^q$
via a learned embedding vector. This is appropriate when one input is unordered and discrete
(e.g., material type, solver choice).

Here we discretize $x_1$ of the Branin function into 5 categories while keeping $x_2$ continuous.

Steps:
1. Setup mlibkriging / jlibkriging
2. Define the Branin function and plot it
3. Build a space-filling design and evaluate it
4. Fit a `WarpKriging` model
5. Predict on a fine grid and plot mean + uncertainty
6. Inspect model parameters

## 0. Setup

Build the C++ core and the Octave binding from source (skip if already built).
Requires: `cmake`, a C++ compiler, Octave ≥ 6.0.

In [1]:
% Add mlibkriging to path
% Adjust the path below to your build/installed directory
repo_root = fullfile(fileparts(pwd()), '..');
build_path = fullfile(repo_root, 'build', 'installed', 'bindings', 'Octave');
if ~exist(fullfile(build_path, 'mLibKriging.mex'), 'file') ...
   && ~exist(fullfile(build_path, ['mLibKriging.', mexext]), 'file')
    error('mlibkriging not found at %s — please build first (see README.md)', build_path);
end
addpath(build_path);
addpath(fullfile(repo_root, 'bindings', 'Octave', 'mlibkriging'));
disp('mlibkriging loaded')

mlibkriging loaded


## 1. Branin function

The Branin function is a standard benchmark for surrogate modelling, defined on $[0,1]^2$
(rescaled from its canonical domain $[-5, 10] \times [0, 15]$).
It has three global minima.

In [1]:
function z = branin(X)
    x1 = X(:,1) * 15 - 5;
    x2 = X(:,2) * 15;
    z = (x2 - 5/(4*pi^2) * x1.^2 + 5/pi * x1 - 6).^2 ...
        + 10 * (1 - 1/(8*pi)) * cos(x1) + 10;
end

% 50x50 evaluation grid
grid_x = linspace(0, 1, 50);
[G1, G2] = meshgrid(grid_x, grid_x);
grid_pts = [G1(:), G2(:)];
z_true = reshape(branin(grid_pts), 50, 50);

figure;
contourf(G1, G2, z_true, 20);
colorbar;
title('True Branin function');
xlabel('x_1'); ylabel('x_2');

## 2. Design of experiments

We discretize $x_1$ into 5 categories (0, 1, ..., 4), each mapped to an
equally spaced center in $[0, 1]$, and sample $n = 40$ points.

In [2]:
rand('seed', 42);
L = 5;  % number of categories
n = 40;
centers = linspace(0.1, 0.9, L);

% x1 is categorical: randomly assign levels
x1_level = randi([0, L-1], n, 1);
x2 = (randperm(n)' - rand(n,1)) / n;

% Evaluate Branin at the level's center for x1
X_eval = [centers(x1_level + 1)', x2];
y = branin(X_eval);

% For WarpKriging, x1 is passed as integer level (0..L-1)
X = [double(x1_level), x2];

figure;
contourf(G1, G2, z_true, 20);
hold on;
scatter(X_eval(:,1), X_eval(:,2), 60, 'w', 'filled', 'MarkerEdgeColor', 'k');
for lvl = 1:L
    line([centers(lvl) centers(lvl)], ylim, 'Color', [0.5 0.5 0.5], 'LineStyle', '--');
end
hold off;
colorbar;
title(sprintf('%d points (%d categories for x_1)', n, L));
xlabel('x_1'); ylabel('x_2');

## 3. Fit a WarpKriging model (`categorical`)

We use `categorical(5,2)` for $x_1$ (5 levels → $\mathbb{R}^2$ embedding) and `kumaraswamy`
for the continuous $x_2$.

In [3]:
wk = WarpKriging(y, X, {'categorical(5,2)', 'kumaraswamy'}, 'matern5_2', 'constant', false, 'Adam', 'LL');
disp(wk.summary())

* WarpKriging
* data: 40x[0,4],[0.0249112,0.995969] -> 40x[2.42113,138.624]
* trend constant (est.): 2212.73
* variance (est.): 2.31782e+06
* covariance:
  * kernel: matern5_2
  * range (est.): 22.631, 1.1794, 7.85903
  * features: 3
  * warpings:
      x0: "categorical(5,2)"  →  Embedding(L=5, q=2)
      x1: "kumaraswamy"  →  Kumaraswamy(a=0.991298, b=0.991689, range=[0.0249112,0.995969])
  * total warp params: 12
  * fit:
    * objective: LL
    * optim: Adam



## 4. Predict and plot

We predict at each level over a dense grid for $x_2$.

In [4]:
x2_grid = linspace(0, 1, 50)';

figure;
for lvl = 0:(L-1)
    X_pred = [repmat(lvl, 50, 1), x2_grid];
    [mu, sd] = wk.predict(X_pred, true, false);

    % True Branin at this level's center
    X_true = [repmat(centers(lvl+1), 50, 1), x2_grid];
    y_true = branin(X_true);

    subplot(1, L, lvl+1);
    plot(x2_grid, y_true, 'k-', 'LineWidth', 1.5); hold on;
    plot(x2_grid, mu, 'b-', 'LineWidth', 1.5);
    fill([x2_grid; flipud(x2_grid)], [mu - 2*sd; flipud(mu + 2*sd)], ...
         'b', 'FaceAlpha', 0.2, 'EdgeColor', 'none');

    mask = (x1_level == lvl);
    scatter(x2(mask), y(mask), 30, 'r', 'filled');
    hold off;
    title(sprintf('Level %d', lvl));
    xlabel('x_2');
    if lvl == 0; ylabel('Branin'); end
end
% (Octave has no sgtitle; per-axes title only)

## 5. Model inspection

Key fitted parameters: length-scales $\theta$, variance $\sigma^2$, log-likelihood, and warping specification.

In [5]:
fprintf('Kernel       : %s\n', wk.kernel());
fprintf('Theta (range): %s\n', mat2str(wk.theta(), 4));
fprintf('Sigma2       : %.4f\n', wk.sigma2());
fprintf('LogLikelihood: %.4f\n', wk.logLikelihood());
fprintf('Feature dim  : %d\n', wk.feature_dim());
fprintf('Warping      : ');
w = wk.warping(); for i = 1:length(w); fprintf('%s ', w{i}); end; fprintf('\n');

Kernel       : matern5_2


Theta (range): [22.63;1.179;7.859]


Sigma2       : 2317818.9366


LogLikelihood: -59.9607


Feature dim  : 3


Warping      : 

categorical(5,2) kumaraswamy 
